# LLM 推理核心模块（Jupyter Notebook 版）

这个 notebook 对应你的 roadmap 前几周实操内容，目标是：

1. 手写并运行 `MHA/GQA`
2. 手写并运行 `RoPE + RMSNorm`
3. 用分块 + online softmax 模拟 `FlashAttention`
4. 手写简化版 `LoRA` 层
5. 实现简化版 `Continuous Batching` 调度器

> 建议运行方式：逐格执行 + 修改参数观察行为变化。

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Optional, List

np.set_printoptions(precision=4, suppress=True)


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x_max = np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x - x_max)
    return ex / np.sum(ex, axis=axis, keepdims=True)


print("Ready: NumPy", np.__version__)

## 1) MHA / GQA（NumPy 实现）

下面实现的是推理前向逻辑，重点看 shape 和 GQA 中 `K/V` 头的复用关系。

In [ ]:
def split_heads(x: np.ndarray, n_heads: int) -> np.ndarray:
    # [B, T, D] -> [B, H, T, Dh]
    b, t, d = x.shape
    assert d % n_heads == 0
    dh = d // n_heads
    return x.reshape(b, t, n_heads, dh).transpose(0, 2, 1, 3)


def merge_heads(x: np.ndarray) -> np.ndarray:
    # [B, H, T, Dh] -> [B, T, D]
    b, h, t, dh = x.shape
    return x.transpose(0, 2, 1, 3).reshape(b, t, h * dh)


def sdpa(q: np.ndarray, k: np.ndarray, v: np.ndarray) -> np.ndarray:
    # q/k/v: [B, H, T, Dh]
    dh = q.shape[-1]
    score = (q @ np.swapaxes(k, -1, -2)) / np.sqrt(dh)
    prob = softmax(score, axis=-1)
    return prob @ v


def mha_gqa_forward(
    x: np.ndarray,
    w_q: np.ndarray,
    w_k: np.ndarray,
    w_v: np.ndarray,
    w_o: np.ndarray,
    n_heads: int,
    n_kv_heads: Optional[int] = None,
) -> np.ndarray:
    if n_kv_heads is None:
        n_kv_heads = n_heads
    assert n_heads % n_kv_heads == 0

    q = x @ w_q
    k = x @ w_k
    v = x @ w_v

    qh = split_heads(q, n_heads)
    kh = split_heads(k, n_kv_heads)
    vh = split_heads(v, n_kv_heads)

    # GQA: 一个 KV 头服务多个 Q 头
    group_size = n_heads // n_kv_heads
    if group_size > 1:
        kh = np.repeat(kh, repeats=group_size, axis=1)
        vh = np.repeat(vh, repeats=group_size, axis=1)

    out = sdpa(qh, kh, vh)
    out = merge_heads(out)
    return out @ w_o

In [ ]:
# Demo: MHA vs GQA
rng = np.random.default_rng(7)
B, T, D = 2, 8, 32
Hq, Hkv = 8, 2

x = rng.normal(size=(B, T, D)).astype(np.float32)
w_q = rng.normal(size=(D, D)).astype(np.float32) * 0.02
w_k = rng.normal(size=(D, D // (Hq // Hkv))).astype(np.float32) * 0.02
w_v = rng.normal(size=(D, D // (Hq // Hkv))).astype(np.float32) * 0.02
w_o = rng.normal(size=(D, D)).astype(np.float32) * 0.02

y = mha_gqa_forward(x, w_q, w_k, w_v, w_o, n_heads=Hq, n_kv_heads=Hkv)
print("GQA output shape:", y.shape)

# MHA 对比（n_kv_heads = n_heads）
w_k_mha = rng.normal(size=(D, D)).astype(np.float32) * 0.02
w_v_mha = rng.normal(size=(D, D)).astype(np.float32) * 0.02
y_mha = mha_gqa_forward(x, w_q, w_k_mha, w_v_mha, w_o, n_heads=Hq, n_kv_heads=Hq)
print("MHA output shape:", y_mha.shape)

## 2) RoPE + RMSNorm

RoPE 作用在 `Q/K` 上，RMSNorm 作用在隐藏状态最后一维。

In [ ]:
def rms_norm(x: np.ndarray, weight: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    assert x.shape[-1] == weight.shape[0]
    rms = np.sqrt(np.mean(x * x, axis=-1, keepdims=True) + eps)
    return (x / rms) * weight


def rotate_half(x: np.ndarray) -> np.ndarray:
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    out = np.empty_like(x)
    out[..., ::2] = -x2
    out[..., 1::2] = x1
    return out


def build_rope_cache(T: int, Dh: int, theta: float = 10000.0):
    assert Dh % 2 == 0
    idx = np.arange(0, Dh, 2, dtype=np.float32)
    inv_freq = 1.0 / (theta ** (idx / Dh))
    pos = np.arange(T, dtype=np.float32)
    freqs = np.outer(pos, inv_freq)  # [T, Dh/2]
    cos = np.repeat(np.cos(freqs), 2, axis=-1)
    sin = np.repeat(np.sin(freqs), 2, axis=-1)
    return cos, sin


def apply_rope(q: np.ndarray, k: np.ndarray, cos: np.ndarray, sin: np.ndarray):
    # q/k: [B, H, T, Dh], cos/sin: [T, Dh]
    _, _, T, Dh = q.shape
    assert cos.shape == (T, Dh) and sin.shape == (T, Dh)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q_out = q * cos + rotate_half(q) * sin
    k_out = k * cos + rotate_half(k) * sin
    return q_out, k_out


# Demo
rng = np.random.default_rng(11)
B, H, T, Dh = 2, 4, 16, 8
q = rng.normal(size=(B, H, T, Dh)).astype(np.float32)
k = rng.normal(size=(B, H, T, Dh)).astype(np.float32)
cos, sin = build_rope_cache(T, Dh)
q2, k2 = apply_rope(q, k, cos, sin)
print("RoPE q/k shape:", q2.shape, k2.shape)

x = rng.normal(size=(B, T, H * Dh)).astype(np.float32)
w = np.ones((H * Dh,), dtype=np.float32)
y = rms_norm(x, w)
print("RMSNorm shape:", y.shape)

## 3) FlashAttention 思路模拟（分块 + Online Softmax）

这里不追求 CUDA 性能，只演示算法核心：分块处理 K/V，并在线更新 softmax 归一化项。

In [ ]:
def flash_attention_tiled(q: np.ndarray, k: np.ndarray, v: np.ndarray, block_size: int = 32):
    # q/k/v: [T, Dh]
    T, Dh = q.shape
    scale = 1.0 / np.sqrt(Dh)
    out = np.zeros_like(q)

    for i in range(0, T, block_size):
        i_end = min(i + block_size, T)
        q_blk = q[i:i_end]  # [Bi, Dh]

        m_i = np.full((q_blk.shape[0],), -np.inf, dtype=q.dtype)
        l_i = np.zeros((q_blk.shape[0],), dtype=q.dtype)
        o_i = np.zeros((q_blk.shape[0], Dh), dtype=q.dtype)

        for j in range(0, T, block_size):
            j_end = min(j + block_size, T)
            k_blk = k[j:j_end]
            v_blk = v[j:j_end]

            s = (q_blk @ k_blk.T) * scale       # [Bi, Bj]
            m_ij = np.max(s, axis=1)            # [Bi]
            p = np.exp(s - m_ij[:, None])       # [Bi, Bj]
            l_ij = np.sum(p, axis=1)            # [Bi]

            m_new = np.maximum(m_i, m_ij)
            alpha = np.exp(m_i - m_new)
            beta = np.exp(m_ij - m_new)
            l_new = alpha * l_i + beta * l_ij

            o_i = (alpha[:, None] * l_i[:, None] * o_i + beta[:, None] * (p @ v_blk)) / l_new[:, None]
            m_i, l_i = m_new, l_new

        out[i:i_end] = o_i

    return out


def attention_ref(q: np.ndarray, k: np.ndarray, v: np.ndarray):
    score = (q @ k.T) / np.sqrt(q.shape[-1])
    prob = softmax(score, axis=-1)
    return prob @ v


# Demo: 对比 reference 与 tiled
rng = np.random.default_rng(19)
T, Dh = 128, 64
q = rng.normal(size=(T, Dh)).astype(np.float32)
k = rng.normal(size=(T, Dh)).astype(np.float32)
v = rng.normal(size=(T, Dh)).astype(np.float32)

y_ref = attention_ref(q, k, v)
y_tile = flash_attention_tiled(q, k, v, block_size=32)
print("shape:", y_tile.shape, "max_abs_err:", float(np.max(np.abs(y_ref - y_tile))))

## 4) LoRA 简化实现

核心公式：`W' = W + (alpha/r) * A @ B`，其中 `W` 冻结，`A/B` 可训练。

In [ ]:
class LoRALinear:
    def __init__(self, in_features: int, out_features: int, rank: int = 8, alpha: float = 16.0, seed: int = 0):
        assert rank > 0
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        rng = np.random.default_rng(seed)
        self.weight = rng.normal(size=(in_features, out_features)).astype(np.float32) * 0.02
        self.lora_a = rng.normal(size=(in_features, rank)).astype(np.float32) * 0.01
        self.lora_b = np.zeros((rank, out_features), dtype=np.float32)

    def delta_weight(self):
        return self.scaling * (self.lora_a @ self.lora_b)

    def merged_weight(self):
        return self.weight + self.delta_weight()

    def forward(self, x: np.ndarray):
        assert x.shape[-1] == self.in_features
        return x @ self.merged_weight()


# Demo
layer = LoRALinear(16, 32, rank=4, alpha=8)
x = np.random.randn(2, 10, 16).astype(np.float32)
y = layer.forward(x)
print("LoRA output shape:", y.shape)
print("||delta_W||:", float(np.linalg.norm(layer.delta_weight())))

## 5) Continuous Batching 调度器（简化版）

策略：优先 decode（延迟敏感），剩余 slot 给 prefill（分块推进）。

In [ ]:
@dataclass
class Request:
    req_id: str
    input_tokens: int
    output_tokens: int
    prefilled: int = 0
    decoded: int = 0
    finished: bool = False

    def stage(self) -> str:
        if self.finished:
            return "done"
        if self.prefilled < self.input_tokens:
            return "prefill"
        return "decode"


class ContinuousBatchScheduler:
    def __init__(self, max_batch_size: int = 8, prefill_chunk: int = 128):
        self.max_batch_size = max_batch_size
        self.prefill_chunk = prefill_chunk
        self.queue: List[Request] = []
        self.time_step = 0

    def submit(self, req: Request):
        self.queue.append(req)

    def active(self):
        return [r for r in self.queue if not r.finished]

    def step(self):
        self.time_step += 1
        active = self.active()
        if not active:
            return

        decode_reqs = [r for r in active if r.stage() == "decode"]
        prefill_reqs = [r for r in active if r.stage() == "prefill"]

        # 1) decode 优先：每个请求每步生成 1 token
        selected_decode = decode_reqs[: self.max_batch_size]
        for r in selected_decode:
            r.decoded += 1
            if r.decoded >= r.output_tokens:
                r.finished = True

        # 2) 剩余 slot 做 prefill chunk
        remain = self.max_batch_size - len(selected_decode)
        if remain > 0:
            selected_prefill = prefill_reqs[:remain]
            for r in selected_prefill:
                r.prefilled = min(r.input_tokens, r.prefilled + self.prefill_chunk)

    def run_until_done(self, max_steps: int = 10_000):
        steps = 0
        while self.active() and steps < max_steps:
            self.step()
            steps += 1
        return steps


# Demo
scheduler = ContinuousBatchScheduler(max_batch_size=4, prefill_chunk=64)
scheduler.submit(Request("r1", input_tokens=512, output_tokens=32))
scheduler.submit(Request("r2", input_tokens=128, output_tokens=64))
scheduler.submit(Request("r3", input_tokens=1024, output_tokens=16))

steps = scheduler.run_until_done()
print("Finished in steps:", steps)
for r in scheduler.queue:
    print(r.req_id, "prefilled=", r.prefilled, "decoded=", r.decoded, "finished=", r.finished)

## 下一步练习建议

1. 把 `mha_gqa_forward` 改成支持 causal mask。
2. 在 FlashAttention 模拟里增加 `causal=True` 分支。
3. 给 LoRA 增加一个最小训练循环（MSE 目标）并观察 `delta_W` 变化。
4. 给调度器增加「最长等待时间优先」策略，与 decode 优先做对比。

如果你愿意，我可以下一步再给你做一个 **KV Cache 专项 notebook**（Paged 分配 + LRU 驱逐 + 命中率统计）。